## **Installing Required Libraries & Datasets**

In [3]:
!pip install datasets

In [4]:
!pip install faiss-cpu

In [5]:
import os

In [6]:
import numpy as np

In [7]:
from sentence_transformers import SentenceTransformer

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
import faiss

In [10]:
from datasets import load_dataset

In [11]:
dataset=load_dataset("CShorten/ML-ArXiv-Papers",split='train')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [12]:
print(dataset)

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [13]:
dataset[0]

{'Unnamed: 0.1': 0,
 'Unnamed: 0': 0.0,
 'title': 'Learning from compressed observations',
 'abstract': '  The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rat

In [14]:
import pandas as pd

In [2]:
!pip install transformers==4.46.3

In [1]:
from transformers import pipeline

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

## **Data Cleaning & Manipulation**

In [15]:
df1=pd.DataFrame(dataset)
df1

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...,...,...
117587,4995,NaN,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,4996,NaN,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,4997,NaN,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,4998,NaN,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [16]:
df1.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'], dtype='object')

In [17]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117592 entries, 0 to 117591
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Unnamed: 0.1  117592 non-null  int64  
 1   Unnamed: 0    112592 non-null  float64
 2   title         117592 non-null  object 
 3   abstract      117592 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 3.6+ MB


In [18]:
df1=df1[['title','abstract']]
df1

,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...
117587,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [19]:
df1.shape

(117592, 2)

In [20]:
df1=df1.head(15000)

In [21]:
df1.shape

(15000, 2)

In [22]:
df1.isnull()

,title,abstract
0,False,False
1,False,False
2,False,False
3,False,False
4,False,False
...,...,...
14995,False,False
14996,False,False
14997,False,False
14998,False,False


In [23]:
df1[df1.isnull()].sum()

,0
title,0
abstract,0


In [24]:
df1['paper_text']=df1["title"]+" "+df1["abstract"]

/tmp/ipykernel_1419/4024907527.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['paper_text']=df1["title"]+" "+df1["abstract"]


In [25]:
df1['paper_text'].head()

,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [26]:
type(df1[['paper_text']])

pandas.core.frame.DataFrame

In [27]:
print(df1["paper_text"].iloc[0])

Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regress

## **Data Embedding**

In [28]:
from sentence_transformers import SentenceTransformer

In [29]:
model=SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [30]:
print(type('model'))

<class 'str'>


In [31]:
sample=df1["paper_text"].iloc[0]
sample

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rate functions. The ideas are\nillustrated on the example of nonparam

In [32]:
df1["paper_text"]=df1["paper_text"].str.replace("\n"," ",regex=False)
df1["paper_text"]=df1["paper_text"].str.strip()

/tmp/ipykernel_1419/2200328001.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["paper_text"]=df1["paper_text"].str.replace("\n"," ",regex=False)
/tmp/ipykernel_1419/2200328001.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["paper_text"]=df1["paper_text"].str.strip()


In [33]:
embeeding=model.encode(sample)

In [34]:
print(type(embeeding))

<class 'numpy.ndarray'>


In [35]:
print(embeeding.shape)

(384,)


In [36]:
embeeding[:50]

array([-0.1315641 , -0.00678266, -0.00367612,  0.03265158,  0.11219642,
        0.01227267,  0.09816719, -0.0900523 ,  0.04231161, -0.01977348,
       -0.03308417,  0.07452948,  0.10632038, -0.02060429, -0.02052106,
        0.00169493,  0.07081953,  0.05854454, -0.11231912,  0.02082474,
        0.05692544,  0.0201578 ,  0.0258311 ,  0.0321703 ,  0.10513764,
       -0.09676763,  0.02700802, -0.0234509 , -0.04549678, -0.01013699,
       -0.01794855, -0.04814427,  0.01077652, -0.03759069,  0.01943481,
        0.03715189,  0.02967844,  0.04330941,  0.04373213,  0.03704866,
       -0.00182594,  0.00455183, -0.00799067,  0.03037368, -0.014378  ,
        0.03795147,  0.0595916 , -0.02583356, -0.06521576,  0.05900268],
      dtype=float32)

In [37]:
sample_embed=model.encode(df1["paper_text"].head(5).to_list())

In [38]:
sample_embed.shape

(5, 384)

In [39]:
from sklearn.metrics.pairwise import cosine_similarity #Dadiii May skip in main code not needed just for intial testing

In [40]:
similarity=cosine_similarity(sample_embed[0].reshape(1,-1),sample_embed[1].reshape(1,-1))
print(similarity)

[[0.36625272]]


In [41]:
for i in range(1,5):
  sim=cosine_similarity(sample_embed[0].reshape(1,-1),sample_embed[i].reshape(1,-1))
  print(sim)

[[0.36625272]]
[[0.33522844]]
[[0.15505108]]
[[0.37421533]]


## **Generate Full embedding**

In [ ]:
if os.path.exists("paper_embeddings.npy"):
  print("Loading existing embedding")
  embeddings=np.load("paper_embeddings.npy")
else:
  print("Generating New")
  embeddings=model.encode(df1["paper_text"].to_list(),batch_size=32,show_progress_bar=True)
  np.save("paper_embeddings.npy", embeddings)
  print("Embeddings saved successfully")

Generating New


Batches:   0%|          | 0/469 [00:00<?, ?it/s]

In [ ]:
print(embeddings.shape)
print(type(embeddings))
embeddings.dtype

## **Normalizing using Faiss**

In [ ]:
import faiss

In [ ]:
if os.path.exists("paper_faiss.index"):
  print("Loading existing")
  index=faiss.read_index("paper_faiss.index")
else:
  print("Creating new faiss index")
  faiss.normalize_L2(embeddings)
  index=faiss.IndexFlatIP(384)
  index.add(embeddings)

  faiss.write_index(index,"paper_faiss.index")
  print("Faiss index saved successfully")

In [ ]:
faiss.normalize_L2(embeddings)

In [ ]:
index=faiss.IndexFlatIP(384)

In [ ]:
index.add(embeddings)

In [ ]:
query="deep learning for medical image analysis"
query_embedding=model.encode([query])
query_embedding.shape

In [ ]:
faiss.normalize_L2(query_embedding)

In [ ]:
D,I=index.search(query_embedding,5)
print(D)
print(I)

In [ ]:
print(df1.iloc[10466]["title"])

In [ ]:
def search_paper(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    for score, idx in zip(D[0], I[0]):
        print("Similarity score", score)
        print("Title", df1.iloc[idx]["title"])
        print("Title", df1.iloc[idx]["abstract"][:500])
        print()

search_paper("deep learning for medical image analysis")

In [ ]:
from transformers import pipeline
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

type(summarizer)

summary = summarizer(df1.iloc[10466]["abstract"], max_length=120, min_length=40)
print(summary)

In [ ]:
def search_paper_summarize(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    for score, idx in zip(D[0], I[0]):
        print("Similarity score", score)
        print("Title", df1.iloc[idx]["title"])
        print("Title", df1.iloc[idx]["abstract"][:500])
        print()
        summary=summarizer(df1.iloc[idx]["abstract"],max_length=120,min_length=40)
        print(summary)
        print(summary[0]["summary_text"])
        print()

search_paper_summarize("deep learning for medical image analysis",5)

In [ ]:
!pip install keybert==0.8.5

In [ ]:
from keybert import KeyBERT

In [ ]:
kw_model=KeyBERT(model)

In [ ]:
text=df1.iloc[10466]["abstract"]
keywrods=kw_model.extract_keywords(text)

In [ ]:
print(type(keywrods))

In [ ]:
keywords=kw_model.extract_keywords(text,keyphrase_ngram_range=(1,3),stop_words="english")

In [ ]:
print(keywords)